In [ ]:
from IPython.display import HTML

HTML(
    '<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/5.15.3/css/all.min.css">'
)

# <i class="fas fa-rocket" style="color:#18ab4b"></i> Endpoint deployment — real-time credit scoring

Train the CatBoost model, then `estimator.deploy(instance_type='local')` spins up the <i class="fas fa-rocket" style="color:#18ab4b"></i> **serving container**. The Predictor scores loan applications and gets back <i class="fas fa-play-circle" style="color:#18ab4b"></i> probability of default and a SHAP-based scorecard score. Mirrors `sagemaker_endpoints/.../sagemaker_endpoint.py` + `inference.py`.

Prereqs: Docker, `pip install 'oblako[sagemaker,notebook]'`.

## <i class="fas fa-user-graduate" style="color:#18ab4b"></i> Build + train

In [ ]:
import os
import tempfile

import creditdata
from sagemaker.estimator import Estimator

from oblako.services import SageMakerService

os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")

IMAGE = "oblako-credit-risk:latest"
HP = {
    "iterations": "150",
    "depth": "6",
    "learning-rate": "0.1",
    "factor": "20",
    "offset": "600",
}

sm = SageMakerService()
sm.build_image(path="container", tag=IMAGE)  # CatBoost BYOC (./container)

data_dir = tempfile.mkdtemp(prefix="credit-train-")
creditdata.write_training_csv(os.path.join(data_dir, "train.csv"), n=600)
out_dir = tempfile.mkdtemp(prefix="credit-out-")

estimator = Estimator(
    image_uri=IMAGE,
    role="arn:aws:iam::000000000000:role/dummy",
    instance_count=1,
    instance_type="local",
    sagemaker_session=sm.get_session(),
    output_path=f"file://{out_dir}",
    hyperparameters=HP,
)
estimator.fit({"train": f"file://{data_dir}"})

## <i class="fas fa-rocket" style="color:#18ab4b"></i> Deploy a local endpoint and <i class="fas fa-play-circle" style="color:#18ab4b"></i> score applications

In [ ]:
from sagemaker.deserializers import JSONDeserializer
from sagemaker.serializers import JSONSerializer

predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="local",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
)
apps = creditdata.sample_applications()
result = predictor.predict({"instances": apps})
for app, pred in zip(apps, result["predictions"]):
    print(
        f"app_score={app['Application_Score']} bureau={app['Bureau_Score']} "
        f"l2i={app['Loan_to_income']:.0f} {app['Residential_Status']:<20} "
        f"-> PD={pred['proba']:.3f} score={pred['score']}"
    )

In [ ]:
predictor.delete_endpoint()
print("endpoint torn down")